In [1]:
import pandas as pd
from openai import OpenAI
import time
import json
import os

# 1. 配置参数
API_KEY = "sk-d39e9fdad7e44688bcec335791c5ada8"
BASE_URL = "https://api.deepseek.com"
INPUT_FILE = "raw_data.csv"  # 你的原始64卦文件
OUTPUT_FILE = "glm4_train_data.csv" # 生成的微调数据
QUESTIONS_PER_GUA = 20 # 每卦生成的问答对数量

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def generate_instruction_data(gua_name, gua_content):
    """
    请求 DeepSeek 生成 20 个问答对
    """
    prompt = f"""
        你是一个精通《周易》的国学大师和专业的数据标注专家。
        请根据以下【原始卦文】内容，生成 {QUESTIONS_PER_GUA} 个不同的用户提问，并针对这些提问给出一个统一、高质量的专业回答。
        
        【要求】：
        1. 提问角度要多样化：包括但不限于直接询问含义、象征意义、深层哲学、对事业/婚恋/决策的指导等。
        2. 提问语气要自然：模拟真实用户在对话框中的问法。
        3. 回答内容：必须基于原始卦文，结合白话文解释、象传、断易天机等内容，整理成一段逻辑通顺、深度专业、格式工整的文字（即 Summary）。
        4. 输出格式：严格输出 JSON 格式，包含一个列表，列表内每个对象有 "content" (提问) 和 "summary" (回答) 两个字段。
        
        【原始卦文】：
        卦名：{gua_name}
        内容：{gua_content}
        """

    max_retries = 3
    for i in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": "你是一个严谨的训练数据合成专家，只输出合法JSON。"},
                    {"role": "user", "content": prompt},
                ],
                response_format={'type': 'json_object'}
            )
            # 解析返回内容
            res_json = json.loads(response.choices[0].message.content)
            # 兼容可能的 key 名字不一致（有些模型喜欢返回 {"data": [...]}）
            if isinstance(res_json, dict):
                for key in res_json:
                    if isinstance(res_json[key], list):
                        return res_json[key]
            return res_json
        except Exception as e:
            print(f"请求失败 {gua_name}, 重试 {i+1}/3... 错误: {e}")
            time.sleep(2)
    return []



# 读取原始数据
if not os.path.exists(INPUT_FILE):
    print(f"未找到输入文件: {INPUT_FILE}，请确保文件名正确。")
    raise("未找到输入文件")

# 假设 CSV 列名是 'name' 和 'content'
df_raw = pd.read_csv(INPUT_FILE)
data_len = df_raw.shape[0]
all_results = []

print(f"开始处理 {data_len} 卦数据，目标：每卦生成 {QUESTIONS_PER_GUA} 条数据...")

for index, row in df_raw.iterrows():
    name = row['挂名'] # 对应你的列名：卦名
    content = row['挂的内容'] # 对应你的列名：卦文内容
    
    print(f"[{index+1}/{data_len}] 正在处理: {name}...")
    
    # 调用 API
    batch_data = generate_instruction_data(name, content)
    
    if batch_data:
        all_results.extend(batch_data)
        print(f"成功生成 {len(batch_data)} 条。")
    else:
        print(f"!!! {name} 处理失败，跳过。")
    
    # 适当控制频率，保护 API
    time.sleep(1)
    
    # break

# 转换为 Pandas 并保存
df_final = pd.DataFrame(all_results)

# 统一列名
df_final = df_final[['content', 'summary']]
df_final = df_final.dropna() #以防意外出现NaN行
df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print("-" * 30)
print(f"所有任务已完成！总共生成 {len(df_final)} 条微调数据。")
print(f"文件已保存至: {OUTPUT_FILE}")



开始处理 20 卦数据，目标：每卦生成 20 条数据...
[1/20] 正在处理: 乾卦...
请求失败 乾卦, 重试 1/3... 错误: Expecting ',' delimiter: line 82 column 3 (char 6391)
成功生成 20 条。
[2/20] 正在处理: 坤卦...
成功生成 20 条。
[3/20] 正在处理: 屯卦...
成功生成 20 条。
[4/20] 正在处理: 蒙卦...
成功生成 20 条。
[5/20] 正在处理: 需卦...
成功生成 20 条。
[6/20] 正在处理: 讼卦...
成功生成 20 条。
[7/20] 正在处理: 师卦...
成功生成 20 条。
[8/20] 正在处理: 比卦...
成功生成 20 条。
[9/20] 正在处理: 小畜卦...
成功生成 20 条。
[10/20] 正在处理: 履卦...
成功生成 21 条。
[11/20] 正在处理: 豫卦...
成功生成 20 条。
[12/20] 正在处理: 否卦...
成功生成 21 条。
[13/20] 正在处理: 同人卦...
成功生成 20 条。
[14/20] 正在处理: 大有卦...
成功生成 20 条。
[15/20] 正在处理: 谦卦...
成功生成 21 条。
[16/20] 正在处理: 豫卦...
成功生成 20 条。
[17/20] 正在处理: 随卦...
成功生成 20 条。
[18/20] 正在处理: 蛊卦...
成功生成 20 条。
[19/20] 正在处理: 临卦...
成功生成 20 条。
[20/20] 正在处理: 观卦...
成功生成 20 条。
------------------------------
所有任务已完成！总共生成 403 条微调数据。
文件已保存至: glm4_train_data.csv


In [2]:
batch_data

[{'content': '观卦的‘盥而不荐’是什么意思？祭祀时为啥不献祭品？',
  'summary': '观卦的核心意象是“观察”与“展示”。卦辞‘盥而不荐，有孚顒若’字面描述祭祀场景：盥指洗手洁面以示虔诚，荐指献祭牺牲；但卦辞强调“不荐”，即仪式简化而心存诚敬。这体现了《周易》重精神轻形式的哲学——内在的诚信（孚）比外在的礼仪更重要。结合《象传》“风行地上”之象，此卦告诫领导者应如风般周流观察民情，以德化民，而非苛求形式。在事业决策中，此卦提示：面对变动需保持冷静观察，暂缓行动；婚恋方面则暗示真诚相待胜过浮华套路，考验阶段需耐心。'},
 {'content': '占卜得到观卦，问事业前景，能给我详细解读一下吗？',
  'summary': '观卦事业解读需结合卦象与爻辞。卦象上巽风下坤地，风行大地喻示观察与传播。事业方面，此卦提示当前环境变动不居，宜多观察、少妄动。卦辞‘盥而不荐’暗示：准备工作要做好，但勿急于求成。邵雍解‘坚守岗位’强调稳守本职，傅佩荣解‘贩卖洋货，须防风险’提示跨领域合作需谨慎。传统解卦‘阴长阳消，正道衰微’并非绝对凶兆，而是提醒：在不利形势下，培养预见力、真诚待人（有孚）可转危为安。总体策略：谦虚谨慎，高瞻远瞩，必要时依附德才兼备者。'},
 {'content': '风地观卦和临卦互为综卦，它们之间有什么关系？',
  'summary': '观卦与临卦互为综卦，卦象颠倒而成：临卦（地泽临）是二阳在下、四阴在上，观卦（风地观）是四阴在下、二阳在上。临卦强调以德临人、主动进取（‘至临’、‘知临’），观卦则强调反观内省、静待时机（‘观我生’、‘观其生’）。二者分别代表动态管理与静态观察，在时序上构成完整循环：先有临卦的主动推进，再有观卦的反思修正。决策中，若得观卦，可联想此前是否过于冒进，需退一步审视全局。'},
 {'content': '我最近感情不顺，抽到观卦，能帮我看看姻缘吗？',
  'summary': '观卦婚恋解读：卦辞‘有孚顒若’强调诚信与庄重。传统解卦明确‘婚恋不顺利，双方应经受住考验，从长计议’。卦象风行地上，喻示感情需如风般自然流动，强求不得。爻辞初六‘童观’警示目光短浅易误判；六二‘窥观’则提示管窥之见不可行。建议：真诚沟通（有孚），保持耐心（观祀观祭），避免因外界压力仓促决定。若对方态度不明，可借‘观我生’